# Session 5: Topic Modeling with LDA
This notebook demonstrates Latent Dirichlet Allocation (LDA) for topic modeling on a small text corpus.
It covers text preprocessing, dictionary and bag-of-words creation, LDA training, and topic interpretation.
For the theory behind LDA, refer to the Session 5 lecture PDF.

In [4]:
doc1 = "I am learning NLP, it is very interesting and exciting. It includes machine learning and deep learning"
doc2 = "My father is a data scientist and he is nlp expert, nlp is ineresting"
doc3 = "My sister has good exposure into android development"
doc4 = "Machine learning and deep learning are the future"
doc5 = "I was rasied in a scientist family"
doc6 = "I love watching anime also listening to music and watching movies"
doc7 = "I love listening to the sound tracks of the movies"
doc_complete = [doc1, doc2, doc3, doc4,doc5,doc6, doc7]

# Install and import libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
import string

# Download required NLTK data one time
# nltk.download('stopwords')
# nltk.download('wordnet')

# Text preprocessing
stop = set(stopwords.words('english'))
exclude = set(string.punctuation)
lemma = WordNetLemmatizer()

def clean(doc):
    stop_free = " ".join([i for i in doc.lower().split() if i not in stop])
    punc_free = ''.join(ch for ch in stop_free if ch not in exclude)
    normalized = " ".join(lemma.lemmatize(word) for word in punc_free.split())
    return normalized

doc_clean = [clean(doc).split() for doc in doc_complete]
print("cleared docs")
print(doc_clean)
print("============================================================")
print()
# Importing gensim (corrected spelling from "genism")
import gensim
from gensim import corpora

# Creating the term dictionary of our corpus
# This line creates a "Mapping Table." 
# It looks at every unique word in the cleaned documents and assigns each one a unique integer ID.
dictionary = corpora.Dictionary(doc_clean)
print("dictionary")
print(dictionary)
print("============================================================")
print()

# In Gensim, doc2bow stands for "document to bag-of-words". 
# It converts each cleaned document (a list of tokens) into a sparse frequency representation:
# [(word_id_1, count_1), (word_id_2, count_2), ...]
# word_id: An integer assigned by dictionary when it was built. Each unique word gets a unique ID (e.g., 0 = "nlp", 1 = "machine", etc.)
# count: How many times that word appears in the current document.
doc_term_matrix = [dictionary.doc2bow(doc) for doc in doc_clean]
# doc_term_matrix
print(doc_term_matrix)
print("============================================================")
print()
# print in more readable way 
for i, doc_bow in enumerate(doc_term_matrix):
    # Convert (id, count) -> (word, count)
    readable_bow = [(dictionary[word_id], count) for word_id, count in doc_bow]
    print(f"Doc {i+1}: {readable_bow}")
print("============================================================")
print()
# Creating and training LDA model
lda_model = gensim.models.ldamodel.LdaModel(
    doc_term_matrix,
    num_topics=3,
    id2word=dictionary, # Maps word IDs to actual words for interpretability.
    passes=50,          # Number of full training iterations over the corpus.
    random_state=100,  # Added for reproducibility
    chunksize=100,     # Added for better performance, Batch size for document processing
    alpha=0.5,     
    eta='auto'         # Let the model learn optimal eta values, β (beta): Gensim simply uses the Greek letter η (eta) for the exact same parameter.
)
# note: chunksize and passes are training efficiency knobs, not model structure parameters.

# random_state=100
# What it does: It sets a "seed" for the random number generator. 
# By picking a specific number (like 100), you ensure that the "random" choices are the exact same every time you run the code.
# Why use it: Without this, you might get slightly different topic results every time you hit "Run."
# This makes it impossible to debug or explain your results to others. Setting a random_state ensures reproducibility.

# chunksize=100 :This controls how many documents the algorithm looks at before it tries to update the model.

# Results
print(lda_model.print_topics())
print("============================================================")
print()

# print in more readable way
# -1 means "show all topics"
for idx, topic in lda_model.print_topics(num_topics=-1):
    print(f"Topic {idx}:")
    
    # 1. Break the string into individual word segments
    parts = topic.split(' + ')
    
    # 2. Extract only the word (the part after the '*') and remove quotes
    # Example: 0.186*"learning" -> learning
    just_words = [p.split('*')[1].replace('"', '').strip() for p in parts]
    
    # 3. Print neatly
    print(f"  Keywords: {', '.join(just_words)}")
    print("-" * 30)


cleared docs
[['learning', 'nlp', 'interesting', 'exciting', 'includes', 'machine', 'learning', 'deep', 'learning'], ['father', 'data', 'scientist', 'nlp', 'expert', 'nlp', 'ineresting'], ['sister', 'good', 'exposure', 'android', 'development'], ['machine', 'learning', 'deep', 'learning', 'future'], ['rasied', 'scientist', 'family'], ['love', 'watching', 'anime', 'also', 'listening', 'music', 'watching', 'movie'], ['love', 'listening', 'sound', 'track', 'movie']]

dictionary
Dictionary<29 unique tokens: ['deep', 'exciting', 'includes', 'interesting', 'learning']...>

[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 3), (5, 1), (6, 1)], [(6, 2), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)], [(12, 1), (13, 1), (14, 1), (15, 1), (16, 1)], [(0, 1), (4, 2), (5, 1), (17, 1)], [(11, 1), (18, 1), (19, 1)], [(20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 2)], [(22, 1), (23, 1), (24, 1), (27, 1), (28, 1)]]

Doc 1: [('deep', 1), ('exciting', 1), ('includes', 1), ('interesting', 1), ('learning', 3),

In [6]:
for i, doc in enumerate(doc_term_matrix):
    topic_dist = lda_model[doc]  # Get topic proportions for the document
    print(topic_dist)
    print()
    print(f"\nDocument {i+1} ({doc_complete[i][:30]}...):")
    for topic_id, prob in topic_dist:
        print(f"  Topic {topic_id}: {prob:.2%}")
    print("====================================================")

[(0, np.float32(0.89689183)), (1, np.float32(0.054949965)), (2, np.float32(0.048158206))]


Document 1 (I am learning NLP, it is very ...):
  Topic 0: 89.69%
  Topic 1: 5.49%
  Topic 2: 4.82%
[(0, np.float32(0.06071239)), (1, np.float32(0.87989455)), (2, np.float32(0.05939303))]


Document 2 (My father is a data scientist ...):
  Topic 0: 6.07%
  Topic 1: 87.99%
  Topic 2: 5.94%
[(0, np.float32(0.8429985)), (1, np.float32(0.07862096)), (2, np.float32(0.078380525))]


Document 3 (My sister has good exposure in...):
  Topic 0: 84.30%
  Topic 1: 7.86%
  Topic 2: 7.84%
[(0, np.float32(0.84474534)), (1, np.float32(0.07767976)), (2, np.float32(0.077574894))]


Document 4 (Machine learning and deep lear...):
  Topic 0: 84.47%
  Topic 1: 7.77%
  Topic 2: 7.76%
[(0, np.float32(0.112019114)), (1, np.float32(0.77570856)), (2, np.float32(0.11227233))]


Document 5 (I was rasied in a scientist fa...):
  Topic 0: 11.20%
  Topic 1: 77.57%
  Topic 2: 11.23%
[(0, np.float32(0.053031962)), (1, np.float3